In [ ]:
import os
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
import math
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# 设置 matplotlib 使用 inline 后端以在 notebook 中显示
%matplotlib inline

# 设置图像目录
ROOT = '/public/home/yunzhuozhang/AWI'
ANADIR = os.path.join(ROOT, 'location_analysis')

# 确保输出目录存在
os.makedirs(ANADIR, exist_ok=True)

In [ ]:
# 文件路径
COMPRESSED_FILES = {
    'fixed': os.path.join(ROOT, 'codarea_mhw_fixedbaseline_compressed.nc'),
    'moving': os.path.join(ROOT, 'codarea_mhw_movingbaseline_compressed.nc'),
}

# 时间参数
TIME_LEN = 37960
NYEARS = 104
NDAY = 365
YEAR0 = 1982

# 温度分组（包含所有组）
BOUNDS = [-math.inf, -0.92, 1.352, 2.119, 2.834, math.inf]
LABELS_FULL = ['< -0.92', 'optimum (-0.92~1.352)', 'Moderate (1.352~2.119)', 'High (2.119~2.834)', 'Severe (>=2.834)']
PLOT_GROUP_INDICES = list(range(len(LABELS_FULL)))  # 包含所有索引 (包括 < -0.92)
LABELS = [LABELS_FULL[i] for i in PLOT_GROUP_INDICES]
NG = len(LABELS)

# 8个位置的命名
LOCATION_NAMES = [
    'Franklin Bay',
    'Western Greenland',
    'Novaya Zemlya',
    'Svalbard',
    'ChukchiSea',
    'East Greenland',
    'Hudson Bay',
    'Severnaya Zemlya'
]

LOCATION_SHORT_NAMES = [
    'L1-R1', 'L1-R2', 'L1-R3', 'L1-R4',
    'L2-R1', 'L2-R2', 'L2-R3', 'L2-R4'
]

# 颜色方案（5个温度组）
# 顺序：< -0.92 (深灰) / optimum (浅蓝) / Moderate (浅黄) / High (橙红) / Severe (深红)
COLORS = [
    (0.35, 0.35, 0.35),  # < -0.92: deeper gray
    (0.45, 0.70, 0.95),  # optimum: deeper light blue
    (0.98, 0.88, 0.35),  # Moderate: deeper light yellow
    (0.95, 0.40, 0.20),  # High: orange-red
    (0.80, 0.10, 0.10),  # Severe: deep red
]

# SST 阶段定义
PHASES = {
    'historical': lambda y: y < 2010,
    'current':    lambda y: 2010 <= y <= 2050,
    'future':     lambda y: y > 2050,
}

PHASE_COLORS = {
    'historical': '#1f77b4',  # blue
    'current':    '#ff7f0e',  # orange
    'future':     '#2ca02c',  # green
}

PHASE_LABELS = {
    'historical': 'historical (1982–2009)',
    'current':    'current (2010–2050)',
    'future':     'future (2051–2085)',
}

# 温度阈值
THRESHOLDS = [-0.92, 1.352, 2.119, 2.834]
SHADE_BANDS = [
    (-np.inf, THRESHOLDS[0], (0.35, 0.35, 0.35), 0.16),
    (THRESHOLDS[0], THRESHOLDS[1], (0.45, 0.70, 0.95), 0.16),
    (THRESHOLDS[1], THRESHOLDS[2], (0.98, 0.88, 0.35), 0.16),
    (THRESHOLDS[2], THRESHOLDS[3], (1.00, 0.27, 0.00), 0.16),
    (THRESHOLDS[3], np.inf, (0.80, 0.10, 0.10), 0.16),
]

print("常量定义完成")

In [ ]:
def process_compressed_data(nc_file):
    """处理压缩的NetCDF文件，计算每个位置的年度统计"""
    print(f"处理文件: {os.path.basename(nc_file)}")
    
    with nc.Dataset(nc_file, 'r') as ds:
        temp_class = ds.variables['temperature_class'][:]
        mhw_class = ds.variables['mhw_class'][:]
        
        if 'original_dim1' in ds.variables:
            dim1_map = ds.variables['original_dim1'][:]
            dim2_map = ds.variables['original_dim2'][:]
        else:
            dim1_map = np.array([0,0,0,0,1,1,1,1])
            dim2_map = np.array([0,1,2,3,0,1,2,3])
        
        print(f"  数据形状: temp_class={temp_class.shape}, mhw_class={mhw_class.shape}")
        
        location_stats = np.zeros((8, NYEARS, len(LABELS_FULL), 2), dtype=np.int64)
        
        for year_idx in range(NYEARS):
            start_day = year_idx * NDAY
            end_day = start_day + NDAY
            
            if end_day > temp_class.shape[1]:
                break
                
            for location in range(8):
                temp_year = temp_class[location, start_day:end_day]
                mhw_year = mhw_class[location, start_day:end_day]
                
                for day in range(min(NDAY, len(temp_year))):
                    temp_group = temp_year[day]
                    mhw_flag = mhw_year[day]
                    
                    if 0 <= temp_group < len(LABELS_FULL) and 0 <= mhw_flag <= 1:
                        location_stats[location, year_idx, temp_group, mhw_flag] += 1
        
        print(f"  统计完成")
        return location_stats, dim1_map, dim2_map

# 加载数据
print("加载 Fixed baseline 数据...")
data_fixed, dim1_map, dim2_map = process_compressed_data(COMPRESSED_FILES['fixed'])

print("\n加载 Moving baseline 数据...")
data_moving, _, _ = process_compressed_data(COMPRESSED_FILES['moving'])

In [ ]:
years = np.arange(YEAR0, YEAR0 + NYEARS)

# 数据选择：只使用所有 5 个温度组
fixed_data_selected = data_fixed[:, :, PLOT_GROUP_INDICES, :]   # (8, 104, 5, 2)
moving_data_selected = data_moving[:, :, PLOT_GROUP_INDICES, :] # (8, 104, 5, 2)

fixed_avg = np.mean(fixed_data_selected, axis=0)   # (104, 5, 2)
moving_avg = np.mean(moving_data_selected, axis=0) # (104, 5, 2)

# 只提取 MHW=1 的数据
fixed_mhw1 = fixed_avg[:, :, 1]  # (104, 5)
moving_mhw1 = moving_avg[:, :, 1]  # (104, 5)

# 定义三个时期
# Historic: 1990-2020
# Current:  2021-2059
# Future:   2060-2085
hist_start = 1990 - YEAR0
hist_end = 2020 - YEAR0 + 1
curr_start = 2021 - YEAR0
curr_end = 2059 - YEAR0 + 1
fut_start = 2060 - YEAR0
fut_end = min(2085 - YEAR0 + 1, NYEARS)

# 计算各时期 Mean 和 Standard Deviation
fixed_hist_mean = np.mean(fixed_mhw1[hist_start:hist_end, :], axis=0)
fixed_curr_mean = np.mean(fixed_mhw1[curr_start:curr_end, :], axis=0)
fixed_fut_mean = np.mean(fixed_mhw1[fut_start:fut_end, :], axis=0)

moving_hist_mean = np.mean(moving_mhw1[hist_start:hist_end, :], axis=0)
moving_curr_mean = np.mean(moving_mhw1[curr_start:curr_end, :], axis=0)
moving_fut_mean = np.mean(moving_mhw1[fut_start:fut_end, :], axis=0)

fixed_hist_std = np.std(fixed_mhw1[hist_start:hist_end, :], axis=0)
fixed_curr_std = np.std(fixed_mhw1[curr_start:curr_end, :], axis=0)
fixed_fut_std = np.std(fixed_mhw1[fut_start:fut_end, :], axis=0)

moving_hist_std = np.std(moving_mhw1[hist_start:hist_end, :], axis=0)
moving_curr_std = np.std(moving_mhw1[curr_start:curr_end, :], axis=0)
moving_fut_std = np.std(moving_mhw1[fut_start:fut_end, :], axis=0)

print(f"fixed_mhw1 形状: {fixed_mhw1.shape}  # 应该是 (104, 5) - MHW=1 数据")
print(f"moving_mhw1 形状: {moving_mhw1.shape}")
print()
print(f"绘制的温度组 (LABELS): {LABELS}")
print(f"温度组数量 (NG): {NG}")

# 定义温度组的简化标签
LABELS_SHORT = ['Cold', 'Optimum', 'Moderate', 'High', 'Severe']

# 字体参数（统一放大，提升整体可读性）
FS_SUPTITLE = 18
FS_TITLE = 14
FS_AXIS = 13
FS_TICK = 11
FS_LEGEND = 11

# 创建图像 (2行2列: Time Series, Mean bar + Std errorbar)
fig = plt.figure(figsize=(19, 11))
gs = fig.add_gridspec(2, 2, width_ratios=[2, 1.2], hspace=0.30, wspace=0.25, bottom=0.26, top=0.90)

legend_lines = []
legend_labels = []

# 计算 y 轴范围（用于时间序列图）
ymax_ts = max(np.nanmax(fixed_mhw1), np.nanmax(moving_mhw1))
if not np.isfinite(ymax_ts) or ymax_ts <= 0:
    ymax_ts = 1.0

# 辅助绘图函数：均值柱 + 标准差误差棒
# 仅绘制“向上”误差（mean -> mean+std），避免触底误差棒的视觉异常。
def plot_grouped_mean_std_bars(ax, mean_groups, std_groups, group_labels, colors, title, ylabel):
    x = np.arange(len(LABELS_SHORT))
    width = 0.25  # 3个柱子，总宽0.75

    offsets = [-width, 0, width]
    ymax_err = 0.0

    for mean_data, std_data, label, color, offset in zip(mean_groups, std_groups, group_labels, colors, offsets):
        mean_data = np.asarray(mean_data, dtype=float)
        std_data = np.asarray(std_data, dtype=float)

        ax.bar(
            x + offset,
            mean_data,
            width,
            label=label,
            color=color,
            alpha=0.86,
            edgecolor='none',
            zorder=2,
        )

        # 上误差棒：不画向下误差，避免底部“线头”
        yerr_upper_only = np.vstack([np.zeros_like(std_data), std_data])
        ax.errorbar(
            x + offset,
            mean_data,
            yerr=yerr_upper_only,
            fmt='none',
            ecolor='black',
            elinewidth=1.05,
            capsize=3,
            capthick=1.05,
            zorder=3,
        )

        # 在误差棒顶部显示“±std”数字
        for xi, yi, si in zip(x + offset, mean_data, std_data):
            if np.isfinite(yi) and np.isfinite(si):
                ax.text(
                    xi,
                    yi + si + 0.02 * max(1.0, np.nanmax(mean_data + std_data)),
                    f"±{si:.1f}",
                    ha='center',
                    va='bottom',
                    fontsize=FS_TICK - 1,
                    color='#333333',
                    rotation=0,
                )

        valid_upper = mean_data + std_data
        if np.isfinite(valid_upper).any():
            ymax_err = max(ymax_err, float(np.nanmax(valid_upper)))

    ax.set_title(title, fontsize=FS_TITLE, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=FS_AXIS)
    ax.set_xticks(x)
    ax.set_xticklabels(LABELS_SHORT, fontsize=FS_TICK)
    ax.tick_params(axis='y', labelsize=FS_TICK)
    ax.set_ylim(0, ymax_err * 1.16 if ymax_err > 0 else 1.0)
    ax.grid(True, alpha=0.25, axis='y', zorder=1)

# 时期配置
period_labels = ['Historic (1990-2020)', 'Current (2021-2059)', 'Future (2060-2085)']
period_colors = ['#1f77b4', '#ff7f0e', '#d62728'] # Blue, Orange, Red

# === 第一行：Fixed Baseline ===
# 1. Time Series
ax_fixed_ts = fig.add_subplot(gs[0, 0])
for gi in range(NG):
    color = COLORS[gi % len(COLORS)]
    line, = ax_fixed_ts.plot(years, fixed_mhw1[:, gi], color=color, linewidth=2.0)
    if gi < len(LABELS_SHORT):
        legend_lines.append(line)
        legend_labels.append(LABELS_SHORT[gi])
ax_fixed_ts.set_title('Fixed Baseline - During MHW Time Series', fontsize=FS_TITLE, fontweight='bold')
ax_fixed_ts.set_ylabel('Days Count', fontsize=FS_AXIS)
ax_fixed_ts.set_ylim(0, ymax_ts * 1.05)
ax_fixed_ts.tick_params(axis='both', labelsize=FS_TICK)
ax_fixed_ts.grid(True, alpha=0.25)

# 2. Mean bar + Std errorbar
ax_fixed_stats = fig.add_subplot(gs[0, 1])
plot_grouped_mean_std_bars(
    ax_fixed_stats,
    [fixed_hist_mean, fixed_curr_mean, fixed_fut_mean],
    [fixed_hist_std, fixed_curr_std, fixed_fut_std],
    period_labels,
    period_colors,
    'Fixed Baseline - Mean with Std Error Bars',
    'Days Count (Mean + Std)',
)

# === 第二行：Moving Baseline ===
# 1. Time Series
ax_moving_ts = fig.add_subplot(gs[1, 0])
for gi in range(NG):
    color = COLORS[gi % len(COLORS)]
    ax_moving_ts.plot(years, moving_mhw1[:, gi], color=color, linewidth=2.0)
ax_moving_ts.set_title('Moving Baseline - During MHW Time Series', fontsize=FS_TITLE, fontweight='bold')
ax_moving_ts.set_xlabel('Year', fontsize=FS_AXIS, labelpad=5)
ax_moving_ts.set_ylabel('Days Count', fontsize=FS_AXIS)
ax_moving_ts.set_ylim(0, ymax_ts * 1.05)
ax_moving_ts.tick_params(axis='both', labelsize=FS_TICK)
ax_moving_ts.grid(True, alpha=0.25)

# 2. Mean bar + Std errorbar
ax_moving_stats = fig.add_subplot(gs[1, 1])
plot_grouped_mean_std_bars(
    ax_moving_stats,
    [moving_hist_mean, moving_curr_mean, moving_fut_mean],
    [moving_hist_std, moving_curr_std, moving_fut_std],
    period_labels,
    period_colors,
    'Moving Baseline - Mean with Std Error Bars',
    'Days Count (Mean + Std)',
)

# 底部双图例：分别居中于左列/右列下方（期刊排版风格）
left_col_box = ax_moving_ts.get_position()
right_col_box = ax_moving_stats.get_position()
left_col_center = 0.5 * (left_col_box.x0 + left_col_box.x1)
right_col_center = 0.5 * (right_col_box.x0 + right_col_box.x1)
legend_y = min(left_col_box.y0, right_col_box.y0) - 0.07

line_legend = fig.legend(
    legend_lines,
    legend_labels,
    loc='upper center',
    ncol=3,
    title='Temperature Categories',
    bbox_to_anchor=(left_col_center, legend_y),
    bbox_transform=fig.transFigure,
    fontsize=FS_LEGEND,
    title_fontsize=FS_LEGEND,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.3,
    borderaxespad=0.0,
)

bar_handles = [Patch(facecolor=c, label=l) for c, l in zip(period_colors, period_labels)]
period_legend = fig.legend(
    handles=bar_handles,
    loc='upper center',
    ncol=1,
    title='Periods',
    bbox_to_anchor=(right_col_center, legend_y),
    bbox_transform=fig.transFigure,
    fontsize=FS_LEGEND,
    title_fontsize=FS_LEGEND,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.0,
    borderaxespad=0.0,
)

fig.add_artist(line_legend)
fig.add_artist(period_legend)

fig.suptitle(
    'During MHW: Trend in Annual Counts | Mean + Standard Deviation (Historic / Current / Future)',
    fontsize=FS_SUPTITLE,
    fontweight='bold',
    y=0.97,
)

filename = "overall_average_time_series.png"
filepath = os.path.join(ANADIR, filename)
fig.savefig(filepath, dpi=170, bbox_inches='tight')
print()
print(f"已保存: {filename}")

plt.show()


In [ ]:
# 加载 SST 数据
def load_sst_arrays(nc_path):
    """加载 SST 和 MHW 数据"""
    with nc.Dataset(nc_path, 'r') as ds:
        sst = ds.variables['sst_all'][:]
        mhw = ds.variables['mhw_class'][:]
        if 'original_dim1' in ds.variables:
            dim1_map = ds.variables['original_dim1'][:]
            dim2_map = ds.variables['original_dim2'][:]
        else:
            dim1_map = np.array([0,0,0,0,1,1,1,1])
            dim2_map = np.array([0,1,2,3,0,1,2,3])
    return sst, mhw, dim1_map, dim2_map

print("加载 SST 数据...")
fixed_sst, fixed_mhw, sst_dim1_map, sst_dim2_map = load_sst_arrays(COMPRESSED_FILES['fixed'])
moving_sst, moving_mhw, _, _ = load_sst_arrays(COMPRESSED_FILES['moving'])

print(f"SST 数据形状: {fixed_sst.shape}")

In [ ]:
def _circular_running_mean(values, window=11):
    """NaN 感知的环形移动平均"""
    if window < 1:
        return values.copy()
    if window % 2 == 0:
        window += 1
    half = window // 2
    v = values.astype(float)
    mask = np.isfinite(v).astype(float)
    v_filled = np.where(np.isfinite(v), v, 0.0)
    
    v_ext = np.concatenate([v_filled[-half:], v_filled, v_filled[:half]])
    m_ext = np.concatenate([mask[-half:], mask, mask[:half]])
    kernel = np.ones(window, dtype=float)
    
    num = np.convolve(v_ext, kernel, mode='valid')
    den = np.convolve(m_ext, kernel, mode='valid')
    with np.errstate(invalid='ignore', divide='ignore'):
        out = num / den
    out[den == 0] = np.nan
    return out

def _fill_nan_circular(y):
    """环形线性插值填补 NaN"""
    x = np.arange(y.size)
    finite = np.isfinite(y)
    if not np.any(finite):
        return y
    xf = x[finite]
    yf = y[finite]
    xf_ext = np.concatenate([xf - y.size, xf, xf + y.size])
    yf_ext = np.concatenate([yf, yf, yf])
    yi = np.interp(x, xf_ext, yf_ext)
    return yi

def compute_phase_month_means(sst, mhw, phase, fill_nan=True):
    """计算给定阶段在每个月上的 SST 平均"""
    years = YEAR0 + np.arange(NYEARS)
    sel_years = np.array([PHASES[phase](int(y)) for y in years], dtype=bool)
    year_ids = np.where(sel_years)[0]
    
    month_days = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dtype=int)
    month_starts = np.concatenate([[0], np.cumsum(month_days)[:-1]])
    
    out = np.full((3, 12), np.nan, dtype=float)
    
    for m in range(12):
        d0 = int(month_starts[m])
        d1 = int(d0 + month_days[m])
        days = np.arange(d0, d1, dtype=int)
        
        if year_ids.size == 0 or days.size == 0:
            continue
        idx = (year_ids[:, None] * NDAY + days[None, :]).reshape(-1)
        idx = idx[(idx >= 0) & (idx < TIME_LEN)]
        if idx.size == 0:
            continue
        
        s_m = sst[idx]
        m_m = mhw[idx]
        
        out[2, m] = np.nanmean(s_m) if s_m.size else np.nan
        
        for f, row in [(0, 0), (1, 1)]:
            cond = (m_m == f)
            out[row, m] = np.nanmean(s_m[cond]) if np.any(cond) else np.nan
    
    if fill_nan:
        for i in range(3):
            yi = _fill_nan_circular(out[i])
            out[i] = yi
    return out

print("SST 辅助函数定义完成")

In [ ]:
# 生成 Fixed / Moving 各自的 3x3 图（Overall + 8 regions），仅显示 During MHW（MHW=1）
# 修改：
# 1. 将每个二十年（首尾为1982-2000及2070-2085）绘制成一根线，以增加样本量减少缺失
# 2. 选用渐变颜色，年份越大颜色越深
# 3. 仅保留 Optimum 温度范围的背景色
# 4. 使用插值填补少量缺失数据
# 5. 内嵌诊断：自动移除数据不足（MHW天数<100 或 季节变化<0.5°C）的曲线

region_names = LOCATION_NAMES
months = np.arange(1, 13)

# 数据质量阈值
MIN_MHW_DAYS = 100       # 最少需要的 MHW=1 天数
MIN_SEASONAL_RANGE = 0.5  # 最小季节变化幅度 (°C)

# 辅助函数：对 NaN 进行环形插值（用于 12 个月的循环数据）
def _fill_nan_circular(arr):
    """对 12 个月数据进行环形插值填补 NaN"""
    arr = np.array(arr, dtype=float)
    if np.all(np.isnan(arr)):
        return arr
    if not np.any(np.isnan(arr)):
        return arr
    
    # 环形扩展
    extended = np.concatenate([arr, arr, arr])
    x_valid = np.where(~np.isnan(extended))[0]
    if len(x_valid) == 0:
        return arr
    y_valid = extended[x_valid]
    
    x_all = np.arange(len(extended))
    extended_filled = np.interp(x_all, x_valid, y_valid)
    
    return extended_filled[12:24]  # 取中间部分

# 按年份区间计算每月均值（带插值）+ 返回数据质量信息
def compute_range_month_means_with_quality(sst, mhw, start_year, end_year):
    """
    计算指定年份区间内每月的 SST 均值，同时返回数据质量信息。
    
    返回: (curve, quality_info)
        curve: 长度为12的数组，MHW=1时的月均SST
        quality_info: dict with 'total_mhw1_days', 'valid_months', 'seasonal_range'
    """
    years = YEAR0 + np.arange(NYEARS)
    sel_years = (years >= start_year) & (years <= end_year)
    year_ids = np.where(sel_years)[0]

    month_days = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dtype=int)
    month_starts = np.concatenate([[0], np.cumsum(month_days)[:-1]])

    out = np.full(12, np.nan, dtype=float)
    monthly_mhw1_counts = np.zeros(12, dtype=int)

    for m in range(12):
        d0 = int(month_starts[m])
        d1 = int(d0 + month_days[m])
        days = np.arange(d0, d1, dtype=int)
        if year_ids.size == 0 or days.size == 0:
            continue
        idx = (year_ids[:, None] * NDAY + days[None, :]).reshape(-1)
        idx = idx[(idx >= 0) & (idx < TIME_LEN)]
        if idx.size == 0:
            continue
        s_m = sst[idx]
        m_m = mhw[idx]
        
        # MHW=1 计算
        cond = (m_m == 1)
        n_mhw1 = np.sum(cond)
        monthly_mhw1_counts[m] = n_mhw1
        if n_mhw1 > 0:
            out[m] = np.nanmean(s_m[cond])
    
    # 计算质量指标（插值前）
    total_mhw1_days = int(np.sum(monthly_mhw1_counts))
    valid_months = int(np.sum(np.isfinite(out)))
    
    # 季节变化幅度（插值前的有效值）
    valid_vals = out[np.isfinite(out)]
    if len(valid_vals) >= 2:
        seasonal_range = float(np.max(valid_vals) - np.min(valid_vals))
    else:
        seasonal_range = 0.0
    
    # 对 NaN 进行插值填补
    if np.any(np.isnan(out)) and not np.all(np.isnan(out)):
        out = _fill_nan_circular(out)
    
    quality_info = {
        'total_mhw1_days': total_mhw1_days,
        'valid_months': valid_months,
        'seasonal_range': seasonal_range
    }
    
    return out, quality_info

# 定义二十年区间（增加样本量以减少缺失数据的影响）
decades = [
    (1982, 2001),   # 20年 (Historic 前半)
    (2002, 2022),   # 22年 (Historic 后半)
    (2023, 2043),   # 21年 (Current/Future 过渡)
    (2044, 2064),   # 21年 (Future 前半)
    (2065, 2085)    # 20年 (Future 后半)
]

# 生成高对比度渐变颜色：从紫色/洋红到深红/橙色
# 使用 plasma colormap，能与浅黄底色形成更好对比
cmap = plt.get_cmap('plasma')
# 在 0.0 到 0.85 之间取样（5条线，避免最亮的黄色部分）
colors = [cmap(i) for i in np.linspace(0.0, 0.85, len(decades))]

# 仅保留 Optimum 背景色带，其他温度区间不着色
# Optimum 改用浅黄色，与紫红渐变线条形成高对比度
NEW_SHADE_BANDS = [
    (THRESHOLDS[0], THRESHOLDS[1], (1.0, 1.0, 0.7), 0.35),    # Optimum: 浅黄色，高对比度
]

for baseline in ['fixed', 'moving']:
    print(f"绘制 {baseline.capitalize()} baseline 的 3x3 图（二十年区间，带插值，自动过滤低质量曲线）...")

    # 预计算所有曲线以便确定 Y 轴范围
    # 结构: all_curves[loc_idx][decade_idx] = curve_data (12,) 或 None（被过滤）
    # 结构: all_quality[loc_idx][decade_idx] = quality_info dict
    all_curves = []
    all_quality = []
    removed_count = 0
    
    # 1. 计算 Overall (作为 loc_idx=0 存储在列表中，后续 loc+1)
    overall_curves = []
    overall_quality = []
    for start_y, end_y in decades:
        # 对该 decade，计算所有 8 个 location 的平均
        loc_means = []
        loc_mhw1_total = 0
        for loc in range(8):
            sst_arr = fixed_sst[loc] if baseline == 'fixed' else moving_sst[loc]
            mhw_arr = fixed_mhw[loc] if baseline == 'fixed' else moving_mhw[loc]
            curve, qinfo = compute_range_month_means_with_quality(sst_arr, mhw_arr, start_y, end_y)
            loc_means.append(curve)
            loc_mhw1_total += qinfo['total_mhw1_days']
        
        # Stack and mean across locations (nanmean 会忽略 NaN)
        decade_mean = np.nanmean(np.stack(loc_means, axis=0), axis=0)
        
        # Overall 的质量评估（使用汇总的 MHW 天数）
        valid_vals = decade_mean[np.isfinite(decade_mean)]
        seasonal_range = float(np.max(valid_vals) - np.min(valid_vals)) if len(valid_vals) >= 2 else 0.0
        
        overall_qinfo = {
            'total_mhw1_days': loc_mhw1_total,
            'valid_months': int(np.sum(np.isfinite(decade_mean))),
            'seasonal_range': seasonal_range
        }
        
        # Overall 通常数据充足，但也检查
        if loc_mhw1_total < MIN_MHW_DAYS or seasonal_range < MIN_SEASONAL_RANGE:
            overall_curves.append(None)
            removed_count += 1
        else:
            overall_curves.append(decade_mean)
        overall_quality.append(overall_qinfo)
    
    all_curves.append(overall_curves)
    all_quality.append(overall_quality)

    # 2. 计算各 Location
    for loc in range(8):
        loc_curves = []
        loc_quality = []
        sst_arr = fixed_sst[loc] if baseline == 'fixed' else moving_sst[loc]
        mhw_arr = fixed_mhw[loc] if baseline == 'fixed' else moving_mhw[loc]
        
        for start_y, end_y in decades:
            curve, qinfo = compute_range_month_means_with_quality(sst_arr, mhw_arr, start_y, end_y)
            
            # 质量检查：移除数据不足的曲线
            if qinfo['total_mhw1_days'] < MIN_MHW_DAYS or qinfo['seasonal_range'] < MIN_SEASONAL_RANGE:
                loc_curves.append(None)  # 标记为移除
                removed_count += 1
            else:
                loc_curves.append(curve)
            loc_quality.append(qinfo)
        
        all_curves.append(loc_curves)
        all_quality.append(loc_quality)

    print(f"  已移除 {removed_count} 条低质量曲线（MHW天数<{MIN_MHW_DAYS} 或 季节变化<{MIN_SEASONAL_RANGE}°C）")

    # 确定 Y 轴范围（仅基于保留的曲线）
    valid_curves = [c for sublist in all_curves for c in sublist if c is not None]
    if len(valid_curves) > 0:
        flat_vals = np.concatenate(valid_curves)
        flat_vals = flat_vals[np.isfinite(flat_vals)]
        if flat_vals.size == 0:
            vmin, vmax = 0.0, 1.0
        else:
            vmin, vmax = np.min(flat_vals), np.max(flat_vals)
    else:
        vmin, vmax = 0.0, 1.0
    pad = 0.05 * (vmax - vmin if (vmax - vmin) > 0 else 1.0)
    ylo, yhi = vmin - pad, vmax + pad

    # 绘图
    fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=True, sharey=True)
    axes = axes.ravel()
    fig.subplots_adjust(hspace=0.16, wspace=0.12, top=0.92, bottom=0.18)

    titles = ['Overall Average'] + [f"{i+1}: {name}" for i, name in enumerate(region_names)]

    for i, ax in enumerate(axes):
        # 背景色带
        for lower, upper, color_band, alpha in NEW_SHADE_BANDS:
            y1 = max(ylo, lower if np.isfinite(lower) else ylo)
            y2 = min(yhi, upper if np.isfinite(upper) else yhi)
            if y2 > y1:
                ax.axhspan(y1, y2, facecolor=color_band, alpha=alpha, zorder=0)
        
        # 绘制曲线（实线，已插值，跳过被移除的曲线）
        curves = all_curves[i]
        for j, curve in enumerate(curves):
            if curve is not None:  # 只绘制通过质量检查的曲线
                label = f"{decades[j][0]}-{decades[j][1]}"
                ax.plot(months, curve, color=colors[j], linewidth=1.8, label=label, zorder=3)
        
        ax.set_title(titles[i])
        ax.set_ylim(ylo, yhi)
        ax.grid(True, alpha=0.18)
        
        # 阈值线
        for thr in THRESHOLDS:
            ax.axhline(thr, color='gray', linestyle='--', linewidth=0.7, alpha=0.5, zorder=2)

        if i % 3 == 0:
            ax.set_ylabel('SST (°C)')
        if i >= 6:
            ax.set_xlabel('Month (1-12)')

    # 底部图例
    # 创建自定义 handles
    legend_handles = [plt.Line2D([0], [0], color=c, lw=2) for c in colors]
    legend_labels = [f"{s}-{e}" for s, e in decades]
    
    fig.legend(legend_handles, legend_labels, loc='lower center', bbox_to_anchor=(0.5, 0.04),
               ncol=5, frameon=True, title='Period (20-year intervals)')

    fig.suptitle(f"{baseline.capitalize()} Baseline - During MHW: SST Trends by 20-Year Periods\n(Low-quality curves with insufficient data removed)",
                 fontsize=14, fontweight='bold')

    outfile = f"sst_timeseries_{baseline}_3x3_20year.png"
    outpath = os.path.join(ANADIR, outfile)
    fig.savefig(outpath, dpi=150, bbox_inches='tight')
    print(f"  已保存: {outfile}")
    plt.show()

print('\nSST 3x3 图已更新：使用二十年区间，自动移除数据不足的曲线。')

In [ ]:
# 绘制新的 temperature group heatmaps
# 优化版：减少重叠，改善布局

NDAY = 365

def compute_daily_fraction(nc_path, years_idx, group_idx):
    """返回形状 (9, NDAY) 的数组：第一行为 overall（locations 平均），其余为 8 个 location。
       值为该 julian day 在所选 years 中被标记为 (mhw==1 and temp_class==group_idx) 的比例（0..100）
    """
    with nc.Dataset(nc_path, 'r') as ds:
        temp_class = ds.variables['temperature_class'][:]
        mhw_class = ds.variables['mhw_class'][:]
    nloc = temp_class.shape[0]
    total_days = temp_class.shape[1]
    max_years = total_days // NDAY
    years_idx = np.array([y for y in years_idx if 0 <= y < max_years], dtype=int)
    n_years = len(years_idx)
    out = np.full((nloc + 1, NDAY), np.nan, dtype=float)
    if n_years == 0:
        return out

    for loc in range(nloc):
        daily_flags = np.zeros((n_years, NDAY), dtype=int)
        for i, y in enumerate(years_idx):
            start = int(y * NDAY)
            end = int(start + NDAY)
            tc = temp_class[loc, start:end]
            mh = mhw_class[loc, start:end]
            cond = (mh == 1) & (tc == group_idx)
            daily_flags[i, :cond.shape[0]] = cond.astype(int)
        frac = daily_flags.mean(axis=0) * 100.0
        out[loc+1, :] = frac
    out[0, :] = np.nanmean(out[1:, :], axis=0)
    return out

# 年份索引
hist_years = np.arange(1990, 2021) - YEAR0
fut_years = np.arange(2060, 2091) - YEAR0

# 定义月份相关参数
month_starts = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335, 365]
month_midpoints = [(month_starts[i] + month_starts[i+1]) / 2 for i in range(12)]
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for baseline_name, nc_path in [('Fixed', COMPRESSED_FILES['fixed']), ('Moving', COMPRESSED_FILES['moving'])]:
    print(f"生成 {baseline_name} baseline 的 heatmaps...")
    
    blocks_hist = []
    blocks_fut = []
    for gi in range(len(LABELS)):
        bh = compute_daily_fraction(nc_path, hist_years, gi)
        bf = compute_daily_fraction(nc_path, fut_years, gi)
        blocks_hist.append(bh)
        blocks_fut.append(bf)

    # 计算全局色标范围
    all_vals = np.concatenate([b.flatten() for b in (blocks_hist + blocks_fut)])
    all_vals = all_vals[np.isfinite(all_vals)]
    if all_vals.size == 0:
        vmin, vmax = 0.0, 1.0
    else:
        vmin = 0.0
        vmax = float(np.nanmax(all_vals))
        if vmax <= vmin:
            vmax = 1.0

    # 绘制 2x5 图 - 不使用 sharey，手动控制每个子图
    fig, axes = plt.subplots(2, len(LABELS), figsize=(24, 8),
                             sharex=False, sharey=False)  # 关键：不共享y轴
    fig.subplots_adjust(hspace=0.45, wspace=0.12,
                       top=0.90, bottom=0.10, left=0.08, right=0.92)

    # Y轴刻度设置
    yticks_pos = np.arange(9) + 0.5  # [0.5, 1.5, ..., 8.5]
    ylabels = ['Overall'] + LOCATION_NAMES  
    
    for col, label in enumerate(LABELS):
        # ============ 上：历史期 ============
        ax = axes[0, col]
        im = ax.imshow(blocks_hist[col], aspect='auto', cmap='YlOrRd',
                       vmin=vmin, vmax=vmax,
                       extent=[1, NDAY+1, 9, 0], interpolation='nearest')
        
        # 子图标题
        ax.set_title(label, fontsize=12, fontweight='bold', pad=15)
        
        # Y轴 - 只在第一列显示标签
        ax.set_ylim(9, 0)
        ax.set_yticks(yticks_pos)
        if col == 0:
            ax.set_ylabel('Historic Period (1990-2020)',
                         fontsize=11, fontweight='bold', labelpad=10)
            ax.set_yticklabels(ylabels, fontsize=9)
        else:
            ax.set_yticklabels([])  # 其他列不显示标签但保留刻度
        
        # X轴：显示月份
        ax.set_xlabel('Month', fontsize=10, fontweight='bold')
        ax.set_xticks(month_midpoints)
        ax.set_xticklabels(month_labels, fontsize=8, rotation=45, ha='right')
        ax.set_xlim(1, NDAY+1)
        
        # 添加月份分隔线
        for ms in month_starts[1:-1]:
            ax.axvline(x=ms, color='white', linewidth=0.8, alpha=0.4)

        # ============ 下：未来期 ============
        ax2 = axes[1, col]
        im2 = ax2.imshow(blocks_fut[col], aspect='auto', cmap='YlOrRd',
                         vmin=vmin, vmax=vmax,
                         extent=[1, NDAY+1, 9, 0], interpolation='nearest')
        
        # Y轴 - 只在第一列显示标签
        ax2.set_ylim(9, 0)
        ax2.set_yticks(yticks_pos)
        if col == 0:
            ax2.set_ylabel('Future Period (2060-2085)',
                          fontsize=11, fontweight='bold', labelpad=10)
            ax2.set_yticklabels(ylabels, fontsize=9)
        else:
            ax2.set_yticklabels([])  # 其他列不显示标签但保留刻度
        
        # X轴：显示儒略日
        ax2.set_xlabel('Julian Day', fontsize=10, fontweight='bold')
        julian_ticks = np.arange(1, 366, 30)
        ax2.set_xticks(julian_ticks)
        ax2.set_xticklabels(julian_ticks, fontsize=9)
        ax2.set_xlim(1, NDAY+1)
        
        # 添加月份分隔线
        for ms in month_starts[1:-1]:
            ax2.axvline(x=ms, color='white', linewidth=0.8, alpha=0.4)

    # ============ Colorbar ============
    cbar_ax = fig.add_axes([0.935, 0.12, 0.012, 0.76])
    cbar = fig.colorbar(im2, cax=cbar_ax, orientation='vertical')
    cbar.set_label('Frequency of MHW days (%)\n(Percentage of years with event)',
                   fontsize=10, fontweight='bold', labelpad=12)
    cbar.ax.tick_params(labelsize=9)

    # ============ 主标题 ============
    fig.suptitle(
        f"{baseline_name} Baseline: Seasonal Distribution of Marine Heatwave Risk Categories\n" +
        f"Top: Historic Period (1990-2020) | Bottom: Future Period (2060-2085)",
        fontsize=14, fontweight='bold', y=1.02
    )

    outname = f"temperature_groups_heatmap_{baseline_name.lower()}_julian.png"
    fig.savefig(os.path.join(ANADIR, outname), dpi=150, bbox_inches='tight')
    print(f"已保存: {outname}")
    plt.show()

print('所有 heatmaps 已生成。')

In [ ]:
# 生成仅包含 Optimum 与 Severe 两组的 heatmaps（其余策略与当前热图一致）
# 列顺序：Optimum, Severe；行：上=Historic (1990-2020)，下=Future (2060-2085)

# 通过标签名鲁棒地确定分组索引
opt_idx = next(i for i, lbl in enumerate(LABELS) if 'optimum' in lbl.lower())
sev_idx = next(i for i, lbl in enumerate(LABELS) if 'severe' in lbl.lower())
sel_indices = [opt_idx, sev_idx]
sel_labels = [LABELS[i] for i in sel_indices]

# 年份索引（与现有热图一致）
hist_years = np.arange(1990, 2021) - YEAR0
fut_years  = np.arange(2060, 2086) - YEAR0  # 截止到 2085（含）

# 使用与当前热图相同的月份轴配置
month_starts = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335, 365]
month_midpoints = [(month_starts[i] + month_starts[i+1]) / 2 for i in range(12)]
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for baseline_name, nc_path in [('Fixed', COMPRESSED_FILES['fixed']), ('Moving', COMPRESSED_FILES['moving'])]:
    print(f"生成 {baseline_name} baseline 的 Optimum/Severe heatmaps...")

    blocks_hist = []
    blocks_fut = []
    for gi in sel_indices:
        bh = compute_daily_fraction(nc_path, hist_years, gi)
        bf = compute_daily_fraction(nc_path, fut_years, gi)
        blocks_hist.append(bh)
        blocks_fut.append(bf)

    # 色标范围（与现有策略一致：全局 0..max）
    all_vals = np.concatenate([b.flatten() for b in (blocks_hist + blocks_fut)])
    all_vals = all_vals[np.isfinite(all_vals)]
    if all_vals.size == 0:
        vmin, vmax = 0.0, 1.0
    else:
        vmin = 0.0
        vmax = float(np.nanmax(all_vals))
        if vmax <= vmin:
            vmax = 1.0

    # 绘制 2x2 图：行=时期, 列=分组（Optimum, Severe）
    fig, axes = plt.subplots(2, len(sel_indices), figsize=(14, 7), sharex=False, sharey=False)
    fig.subplots_adjust(hspace=0.45, wspace=0.16, top=0.90, bottom=0.12, left=0.10, right=0.90)

    yticks_pos = np.arange(9) + 0.5
    ylabels = ['Overall'] + LOCATION_NAMES

    for col, label in enumerate(sel_labels):
        # 上：历史期
        ax = axes[0, col]
        im = ax.imshow(blocks_hist[col], aspect='auto', cmap='YlOrRd',
                       vmin=vmin, vmax=vmax,
                       extent=[1, NDAY+1, 9, 0], interpolation='nearest')
        ax.set_title(label, fontsize=12, fontweight='bold', pad=10)
        ax.set_ylim(9, 0)
        ax.set_yticks(yticks_pos)
        if col == 0:
            ax.set_ylabel('Historic Period (1990-2020)', fontsize=11, fontweight='bold', labelpad=10)
            ax.set_yticklabels(ylabels, fontsize=9)
        else:
            ax.set_yticklabels([])
        ax.set_xlabel('Month', fontsize=10, fontweight='bold')
        ax.set_xticks(month_midpoints)
        ax.set_xticklabels(month_labels, fontsize=8, rotation=45, ha='right')
        ax.set_xlim(1, NDAY+1)
        for ms in month_starts[1:-1]:
            ax.axvline(x=ms, color='white', linewidth=0.8, alpha=0.4)

        # 下：未来期
        ax2 = axes[1, col]
        im2 = ax2.imshow(blocks_fut[col], aspect='auto', cmap='YlOrRd',
                         vmin=vmin, vmax=vmax,
                         extent=[1, NDAY+1, 9, 0], interpolation='nearest')
        ax2.set_ylim(9, 0)
        ax2.set_yticks(yticks_pos)
        if col == 0:
            ax2.set_ylabel('Future Period (2060-2085)', fontsize=11, fontweight='bold', labelpad=10)
            ax2.set_yticklabels(ylabels, fontsize=9)
        else:
            ax2.set_yticklabels([])
        ax2.set_xlabel('Julian Day', fontsize=10, fontweight='bold')
        julian_ticks = np.arange(1, 366, 30)
        ax2.set_xticks(julian_ticks)
        ax2.set_xticklabels(julian_ticks, fontsize=9)
        ax2.set_xlim(1, NDAY+1)
        for ms in month_starts[1:-1]:
            ax2.axvline(x=ms, color='white', linewidth=0.8, alpha=0.4)

    # Colorbar
    cbar_ax = fig.add_axes([0.92, 0.14, 0.012, 0.72])
    cbar = fig.colorbar(im2, cax=cbar_ax, orientation='vertical')
    cbar.set_label('Frequency of MHW days (%)\n(Percentage of years with event)', fontsize=10, fontweight='bold', labelpad=12)
    cbar.ax.tick_params(labelsize=9)

    # 标题
    fig.suptitle(
        f"{baseline_name} Baseline: Seasonal Distribution (Optimum & Severe Only)\n" +
        f"Top: Historic Period (1990-2020) | Bottom: Future Period (2060-2085)",
        fontsize=14, fontweight='bold', y=1.02
    )

    outname = f"temperature_groups_heatmap_{baseline_name.lower()}_opt_severe.png"
    fig.savefig(os.path.join(ANADIR, outname), dpi=150, bbox_inches='tight')
    print(f"已保存: {outname}")
    plt.show()

In [ ]:
# 合并 Optimum 与 Severe 两组的 heatmaps（仅 Moving Baseline）
# 布局为 2 行 × 3 列：
#   行：第 1 行 Optimum，第 2 行 Severe
#   列：Historic (1990-2020), Current (2021-2059), Future (2060-2085)
# 修改：
# 1. 将冬春季居中显示（Julian Day 顺序：从7月开始到次年6月）
# 2. 使用百分比频率（0-20%），超过20%的值用最深色表示
# 3. 每天一个格子（日分辨率）
# 4. 使用分类风格的 colormap

from matplotlib.colors import BoundaryNorm, ListedColormap

# 通过标签名鲁棒地确定分组索引
opt_idx = next(i for i, lbl in enumerate(LABELS) if 'optimum' in lbl.lower())
sev_idx = next(i for i, lbl in enumerate(LABELS) if 'severe' in lbl.lower())

# 定义三个时期的年份索引
hist_years = np.arange(1990, 2021) - YEAR0      # Historic: 1990-2020
curr_years = np.arange(2021, 2060) - YEAR0      # Current: 2021-2059
fut_years  = np.arange(2060, 2086) - YEAR0      # Future: 2060-2085

periods = [
    ('Historic (1990-2020)', hist_years),
    ('Current (2021-2059)', curr_years),
    ('Future (2060-2085)', fut_years),
]

def compute_daily_fraction_v2(nc_path, years_idx, group_idx):
    """计算每个位置每天的 MHW 发生频率（百分比 0-100）
    返回形状 (9, 365) 的数组：第一行为 overall（locations 平均），其余为 8 个 location
    值 = (该天在所选年份中发生 MHW 的年数 / 总年数) * 100
    
    含义：例如 20% 表示在该时期内，有 20% 的年份在这一天发生了指定温度组的 MHW
    """
    with nc.Dataset(nc_path, 'r') as ds:
        temp_class = ds.variables['temperature_class'][:]
        mhw_class = ds.variables['mhw_class'][:]
    nloc = temp_class.shape[0]
    total_days = temp_class.shape[1]
    max_years = total_days // NDAY
    years_idx = np.array([y for y in years_idx if 0 <= y < max_years], dtype=int)
    n_years = len(years_idx)
    
    out = np.full((nloc + 1, NDAY), np.nan, dtype=float)
    if n_years == 0:
        return out
    
    for loc in range(nloc):
        # 使用向量化方式加速
        daily_flags = np.zeros((n_years, NDAY), dtype=int)
        for i, y in enumerate(years_idx):
            start = int(y * NDAY)
            end = int(start + NDAY)
            if end > total_days:
                continue
            tc = temp_class[loc, start:end]
            mh = mhw_class[loc, start:end]
            cond = (mh == 1) & (tc == group_idx)
            daily_flags[i, :len(cond)] = cond.astype(int)
        
        # 计算每天的发生频率（百分比）
        frac = daily_flags.mean(axis=0) * 100.0
        out[loc + 1, :] = frac
    
    # Overall: 各 location 的平均
    out[0, :] = np.nanmean(out[1:, :], axis=0)
    return out

def reorder_to_winter_center(data_2d):
    """将数据重排为从7月开始，使冬春季居中
    输入: (n_rows, 365) 数组
    输出: (n_rows, 365) 数组，但顺序变为 Jul-Dec, Jan-Jun
    """
    jul1_idx = 181  # Jul 1 = day 182 (0-indexed: 181)
    return np.concatenate([data_2d[:, jul1_idx:], data_2d[:, :jul1_idx]], axis=1)

# 重排后的月份起始位置（从7月开始）
month_starts_reordered = [1, 32, 63, 93, 124, 154, 185, 216, 244, 275, 305, 336, 366]
month_midpoints_reordered = [(month_starts_reordered[i] + month_starts_reordered[i+1]) / 2 for i in range(12)]
month_labels_reordered = ['Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 
                          'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']

# 冬春季高亮区域（Jan-May 在新坐标系中的位置）
winter_spring_start = 185  # Jan 1
winter_spring_end = 336    # Jun 1 (end of May)

# 仅使用 Moving Baseline
baseline_name = 'Moving'
nc_path = COMPRESSED_FILES['moving']

print(f"准备 {baseline_name} baseline 的 Optimum/Severe 日数据块（冬春居中，三个时期）...")

# 计算数据块：blocks[group][period_idx] = data array (9, 365)
blocks = {
    'Optimum': [],
    'Severe': []
}

for period_name, years_idx in periods:
    # Optimum
    data_opt = compute_daily_fraction_v2(nc_path, years_idx, opt_idx)
    blocks['Optimum'].append(reorder_to_winter_center(data_opt))
    
    # Severe
    data_sev = compute_daily_fraction_v2(nc_path, years_idx, sev_idx)
    blocks['Severe'].append(reorder_to_winter_center(data_sev))

# 计算全局数据范围
all_vals = np.concatenate([arr.flatten() for g in ['Optimum', 'Severe'] for arr in blocks[g]])
all_vals = all_vals[np.isfinite(all_vals)]
data_max = float(np.nanmax(all_vals)) if all_vals.size > 0 else 100.0

print(f"  数据范围: 0 - {data_max:.1f}% (MHW 发生频率)")
print(f"  注：百分比表示在该时期内，有多少比例的年份在某天发生了指定温度组的 MHW")

# 定义分类颜色映射（范围 0-20%，超过 20% 用最深色）
# 使用规整的整数边界：0, 2, 4, 6, 8, 10, 15, 20
category_bounds = [0, 2, 4, 6, 8, 10, 15, 20, max(21, data_max + 1)]
n_categories = len(category_bounds) - 1  # 8 个分类

# 使用 YlOrRd 的离散版本
base_cmap = plt.get_cmap('YlOrRd')
category_colors = [base_cmap(i / (n_categories - 1)) for i in range(n_categories)]
cat_cmap = ListedColormap(category_colors)
cat_norm = BoundaryNorm(category_bounds, cat_cmap.N)

# 布局：2 行（Optimum/Severe）× 3 列（Historic/Current/Future）
fig, axes = plt.subplots(2, 3, figsize=(20, 10), sharex=True, sharey=False)
fig.subplots_adjust(hspace=0.20, wspace=0.12, top=0.90, bottom=0.12, left=0.10, right=0.90)

# Y轴刻度设置（与其他热图一致）
yticks_pos = np.arange(9) + 0.5  # [0.5, 1.5, ..., 8.5]
ylabels = ['Overall'] + LOCATION_NAMES  # 完整的位置标签

row_labels = ['Optimum', 'Severe']
group_keys = ['Optimum', 'Severe']

# 存储最后一个 imshow 用于 colorbar
last_im = None

for r, (row_label, gkey) in enumerate(zip(row_labels, group_keys)):
    for c, (period_name, _) in enumerate(periods):
        ax = axes[r, c]
        arr = blocks[gkey][c]  # shape (9, 365)
        
        # 使用分类颜色映射
        im = ax.imshow(arr, aspect='auto', cmap=cat_cmap, norm=cat_norm,
                       extent=[1, NDAY+1, 9, 0], interpolation='nearest')
        last_im = im
        
        # 标题：只在第一行显示时期
        if r == 0:
            ax.set_title(period_name, fontsize=13, fontweight='bold', pad=10)
        
        ax.set_ylim(9, 0)
        ax.set_yticks(yticks_pos)
        
        # Y轴标签：只在第一列显示（与其他热图格式一致）
        if c == 0:
            ax.set_ylabel(f'{row_label} Temperature Group',
                         fontsize=11, fontweight='bold', labelpad=10)
            ax.set_yticklabels(ylabels, fontsize=9)
        else:
            ax.set_yticklabels([])  # 其他列不显示标签但保留刻度
        
        # X轴标签：只在最后一行显示
        ax.set_xticks(month_midpoints_reordered)
        if r == 1:
            ax.set_xlabel('Month', fontsize=11, fontweight='bold')
            ax.set_xticklabels(month_labels_reordered, fontsize=9, rotation=45, ha='right')
        else:
            ax.set_xticklabels([])
        
        ax.set_xlim(1, NDAY+1)
        
        # 添加月份分隔线
        for ms in month_starts_reordered[1:-1]:
            ax.axvline(x=ms, color='white', linewidth=0.8, alpha=0.5)
        
        # 高亮冬春季区域（Jan-May）
        ax.axvspan(winter_spring_start, winter_spring_end, 
                   facecolor='none', edgecolor='#2166AC', linewidth=2.5, 
                   linestyle='-', alpha=0.9, zorder=5)

# 在图底部添加冬春季说明
fig.text(0.5, 0.02, 'Blue box highlights Winter-Spring period (Jan-May)', 
         ha='center', fontsize=10, fontstyle='italic', color='#2166AC')

# 分类风格 Colorbar - 使用边界值作为 ticks（规整整数）
cbar_ax = fig.add_axes([0.92, 0.12, 0.020, 0.76])
cbar_ticks = category_bounds[:-1]  # 显示边界值：0, 2, 4, 6, 8, 10, 15, 20
cbar = fig.colorbar(last_im, cax=cbar_ax, orientation='vertical', ticks=cbar_ticks)
cbar.ax.set_yticklabels([f'{int(t)}%' for t in cbar_ticks], fontsize=9)
cbar.set_label('MHW Occurrence Frequency\n(% of Years)', 
               fontsize=11, fontweight='bold', labelpad=12)

fig.suptitle('Moving Baseline: Daily MHW Occurrence by Temperature Category\n(Winter-Spring Centered View)', 
             fontsize=15, fontweight='bold', y=0.98)

outname = 'temperature_groups_heatmap_moving_daily_categories.png'
fig.savefig(os.path.join(ANADIR, outname), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 极坐标圆形北极地图（最外侧为 50°N）
# 目标：
# 1) 高亮不再是黑块：不用 hatch，用清晰半透明填充
# 2) 高亮只出现在海洋：用 NaturalEarth 陆地几何做差集（抠陆地）
# 3) 绿色边界只沿海洋部分出现：对 ocean_poly 的边界描边（本版改为实线）
# 4) 区域名称移到地图外：引线从高亮区域边缘(朝外侧)出发，穿过陆地也可见，一直连到圆外标注
# 5) 经纬度网格线也显示在陆地之上

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path


def _set_circular_boundary(ax):
    theta = np.linspace(0, 2 * np.pi, 400)
    verts = np.column_stack([0.5 + 0.5 * np.cos(theta), 0.5 + 0.5 * np.sin(theta)])
    circle = Path(verts)
    ax.set_boundary(circle, transform=ax.transAxes)


def _wrap_lon_180(lon):
    lon = float(lon)
    if lon > 180:
        lon -= 360
    if lon < -180:
        lon += 360
    return lon


def _box_polygons(lon_w, lon_e, lat_s, lat_n):
    """返回一个或两个 lon/lat 多边形（跨日期变更线时拆分）。"""
    lon_w = _wrap_lon_180(lon_w)
    lon_e = _wrap_lon_180(lon_e)

    def poly(l0, l1):
        return ([l0, l1, l1, l0], [lat_s, lat_s, lat_n, lat_n])

    if lon_w <= lon_e:
        return [poly(lon_w, lon_e)]
    return [poly(lon_w, 180), poly(-180, lon_e)]


def _iter_lonlat_lines_from_boundary(geom):
    """将 shapely 的 boundary 转成可绘制的若干 (lons, lats) 线段列表。"""
    import shapely.geometry as sgeom

    boundary = geom.boundary

    def _line_to_xy(line):
        x, y = line.xy
        return list(x), list(y)

    lines = []
    if isinstance(boundary, sgeom.LineString):
        lines.append(_line_to_xy(boundary))
    elif isinstance(boundary, sgeom.MultiLineString):
        for line in boundary.geoms:
            lines.append(_line_to_xy(line))
    else:
        pass

    return lines


def _centroid_lonlat(geom):
    """返回几何的 (lon, lat) 质心坐标（PlateCarree 下使用）。"""
    c = geom.centroid
    return float(c.x), float(c.y)


def _axes_center_lonlat(ax):
    """获取当前地图视域中心（lon/lat）。"""
    import cartopy.crs as ccrs

    lon0, lon1, lat0, lat1 = ax.get_extent(crs=ccrs.PlateCarree())
    return (lon0 + lon1) / 2.0, (lat0 + lat1) / 2.0


def _raise_gridlines_zorder(gl, zorder):
    """尽可能把 gridlines（及其标签）抬到更高 zorder。"""
    try:
        gl.zorder = zorder
    except Exception:
        pass
    for attr in ("xline_artists", "yline_artists", "xlabel_artists", "ylabel_artists"):
        artists = getattr(gl, attr, None)
        if not artists:
            continue
        for a in artists:
            try:
                a.set_zorder(zorder)
            except Exception:
                pass


def _radial_outer_boundary_point_axes(ax, lon, lat):
    """给定一个区域内部点(lon,lat)，沿“中心->外侧”方向，返回该区域在该方向上的边界点(轴坐标)。
    以轴坐标距离中心的半径最大为准，得到更贴近区域外缘的引线起点。"""
    import cartopy.crs as ccrs
    import shapely.geometry as sgeom

    # 该点在轴坐标
    xA, yA = ax.transAxes.inverted().transform(
        ax.transData.transform(ax.projection.transform_point(lon, lat, ccrs.PlateCarree()))
    )
    if not (np.isfinite(xA) and np.isfinite(yA)):
        return None
    dx, dy = xA - 0.5, yA - 0.5
    dn = (dx * dx + dy * dy) ** 0.5
    if not np.isfinite(dn) or dn == 0:
        return None
    dx, dy = dx / dn, dy / dn

    # 扫描区域边界，取投影到( dx,dy )方向上最外侧的点
    ocean_boundary = getattr(ax, "_tmp_ocean_boundary", None)
    if ocean_boundary is None:
        return None
    boundary = ocean_boundary
    best = None
    best_proj = -1e30

    def consider_xy(x, y):
        nonlocal best, best_proj
        px, py = x - 0.5, y - 0.5
        proj = px * dx + py * dy
        if proj > best_proj:
            best_proj = proj
            best = (x, y)

    # 边界是 LineString/MultiLineString（在轴坐标里）
    if isinstance(boundary, sgeom.LineString):
        xs, ys = boundary.xy
        for x, y in zip(xs, ys):
            consider_xy(float(x), float(y))
    elif isinstance(boundary, sgeom.MultiLineString):
        for line in boundary.geoms:
            xs, ys = line.xy
            for x, y in zip(xs, ys):
                consider_xy(float(x), float(y))
    else:
        return None

    return best


def _ocean_boundary_to_axes(ax, ocean_geom):
    """把 ocean_geom.boundary 从 lon/lat 转到轴坐标(0..1)下的 LineString/MultiLineString。"""
    import cartopy.crs as ccrs
    import shapely.geometry as sgeom

    segs = []
    for seg_lons, seg_lats in _iter_lonlat_lines_from_boundary(ocean_geom):
        pts = []
        for lo, la in zip(seg_lons, seg_lats):
            x, y = ax.transAxes.inverted().transform(
                ax.transData.transform(ax.projection.transform_point(float(lo), float(la), ccrs.PlateCarree()))
            )
            if np.isfinite(x) and np.isfinite(y):
                pts.append((float(x), float(y)))
        if len(pts) >= 2:
            segs.append(sgeom.LineString(pts))
    if not segs:
        return None
    if len(segs) == 1:
        return segs[0]
    return sgeom.MultiLineString(segs)


def _add_outside_label_with_leader(ax, lon, lat, text, color, r_label=0.56, zorder=60):
    """从 (lon,lat) 附近（会自动吸附到高亮区域外缘）画引线到圆外标注。
    线段用轴坐标绘制，确保不受投影/裁剪影响且能覆盖在陆地之上。"""
    import cartopy.crs as ccrs

    # 默认起点：该点在轴坐标
    xA0, yA0 = ax.transAxes.inverted().transform(
        ax.transData.transform(ax.projection.transform_point(lon, lat, ccrs.PlateCarree()))
    )
    if not (np.isfinite(xA0) and np.isfinite(yA0)):
        return

    # 方向：中心 -> 起点
    dx, dy = xA0 - 0.5, yA0 - 0.5
    dn = (dx * dx + dy * dy) ** 0.5
    if not np.isfinite(dn) or dn == 0:
        return
    dx, dy = dx / dn, dy / dn

    # 圆边交点、文本点
    xB, yB = 0.5 + 0.5 * dx, 0.5 + 0.5 * dy
    xT, yT = 0.5 + r_label * dx, 0.5 + r_label * dy

    # 起点吸附：用区域边界在该方向的最外侧点
    best = _radial_outer_boundary_point_axes(ax, lon, lat)
    if best is not None:
        xA, yA = best
    else:
        xA, yA = xA0, yA0

    # 引线：从区域外缘 -> 圆边 -> 圆外
    ax.plot([xA, xB, xT], [yA, yB, yT], transform=ax.transAxes, color=color, linewidth=1.6,
            linestyle=(0, (4, 3)), zorder=zorder, clip_on=False)

    ha = 'left' if xT >= 0.5 else 'right'
    ax.text(xT, yT, text, transform=ax.transAxes, ha=ha, va='center', fontsize=9, color=color,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.9, edgecolor='none'),
            zorder=zorder + 1, clip_on=False)


def plot_arctic_map_outer_50N_with_ocean_only_boxes():
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.io import shapereader
    import shapely.geometry as sgeom
    from shapely.ops import unary_union

    proj = ccrs.NorthPolarStereo()
    fig = plt.figure(figsize=(8.6, 8.6))
    ax = plt.axes(projection=proj)

    ax.set_title('Arctic Map (Circular) | Ocean-only Highlights | Outer Boundary = 50°N',
                 fontsize=12, fontweight='bold')

    ax.set_extent([-180, 180, 50, 90], crs=ccrs.PlateCarree())

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.6,
        color='gray',
        alpha=0.55,
        linestyle='--',
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = False
    gl.xlocator = plt.FixedLocator(np.arange(-180, 181, 30))
    gl.ylocator = plt.FixedLocator(np.arange(50, 91, 10))

    lons = np.linspace(-180, 180, 721)
    ax.plot(lons, np.full_like(lons, 50.0), transform=ccrs.PlateCarree(),
            color='black', linewidth=2.8, zorder=40)

    land_shp = shapereader.natural_earth(resolution='110m', category='physical', name='land')
    land_geoms = list(shapereader.Reader(land_shp).geometries())
    land_union = unary_union(land_geoms)

    boxes = [
        ('Franklin Bay',      -128, -125, 69, 72),
        ('Western Greenland',  -54,  -51, 70, 73),
        ('Novaya Zemlya',       30,   70, 63, 76),
        ('Svalbard',            15,   30, 76, 81),
        ('Chukchi Sea',       -178, -160, 67, 72),
        ('East Greenland',     -40,  -15, 67, 76),
        ('Hudson Bay',         -95,  -74, 51, 64),
        ('Severnaya Zemlya',    90,  108, 77, 82),
    ]

    fill_color = '#2dd4bf'
    edge_color = '#0f766e'

    ocean_geoms_by_name = {}

    for name, lon_w, lon_e, lat_s, lat_n in boxes:
        ocean_parts = []
        for lons_poly, lats_poly in _box_polygons(lon_w, lon_e, lat_s, lat_n):
            box_poly = sgeom.Polygon(zip(lons_poly, lats_poly))
            ocean_poly = box_poly.difference(land_union)
            if ocean_poly.is_empty:
                continue
            ocean_parts.append(ocean_poly)
            ax.add_geometries(
                [ocean_poly],
                crs=ccrs.PlateCarree(),
                facecolor=fill_color,
                edgecolor='none',
                alpha=0.35,
                zorder=3,
            )
        if ocean_parts:
            ocean_geoms_by_name[name] = unary_union(ocean_parts)

    ax.add_feature(cfeature.LAND.with_scale('110m'), facecolor='#d9d9d9', edgecolor='none', zorder=6)
    ax.add_feature(cfeature.COASTLINE.with_scale('110m'), linewidth=0.9, zorder=7)

    _raise_gridlines_zorder(gl, zorder=55)

    for name, lon_w, lon_e, lat_s, lat_n in boxes:
        ocean_geom = ocean_geoms_by_name.get(name)
        if ocean_geom is None or ocean_geom.is_empty:
            continue

        for seg_lons, seg_lats in _iter_lonlat_lines_from_boundary(ocean_geom):
            ax.plot(
                seg_lons,
                seg_lats,
                transform=ccrs.PlateCarree(),
                color=edge_color,
                linewidth=2.6,
                linestyle='-',
                zorder=50,
            )

        # 为“吸附到外缘”准备：把当前海洋几何边界缓存为轴坐标线几何
        ax._tmp_ocean_boundary = _ocean_boundary_to_axes(ax, ocean_geom)

        # 引线起点：沿中心指向外侧的方向，吸附到海洋边界外缘点
        lon_c, lat_c = _centroid_lonlat(ocean_geom)
        _add_outside_label_with_leader(ax, lon_c, lat_c, name, edge_color, r_label=0.58, zorder=60)

    # 清理临时属性
    if hasattr(ax, "_tmp_ocean_boundary"):
        delattr(ax, "_tmp_ocean_boundary")

    _set_circular_boundary(ax)
    return fig, ax


fig, ax = plot_arctic_map_outer_50N_with_ocean_only_boxes()
plt.show()

In [ ]:
# === New: Per-location heatmap | Moving baseline daily MHW Occurrence Frequency ===
# 每个海区单独一张图：Optimum vs Severe 并排；y=五个年份段；每格为“该年份段内每年同一天发生的占比（% of years）”
# 额外：每个海区使用自定义“中心季节”来重排 x 轴，并用线条高亮该月份窗口（不使用任何半透明背景）

import numpy as np
import matplotlib.pyplot as plt


def _daily_occurrence_frequency_percent(nc_path, years_idx, group_idx):
    """返回形状 (9, 365) 的百分比数组：行=Overall + 8 locations；列=day 1..365。
    口径对齐旧图（`compute_daily_fraction_v2`）：对每个日序 d，用年份数作分母。
    """
    import netCDF4 as nc
    import numpy as np

    years_idx = np.asarray(years_idx, dtype=int)
    years_idx = years_idx[(years_idx >= 0) & (years_idx < NYEARS)]
    if years_idx.size == 0:
        return np.zeros((9, NDAY), dtype=float)

    with nc.Dataset(nc_path, 'r') as ds:
        mhw_class = ds.variables['mhw_class'][:]              # (8, TIME_LEN)
        temp_class = ds.variables['temperature_class'][:]     # (8, TIME_LEN)

    out = np.zeros((9, NDAY), dtype=float)
    for loc in range(8):
        acc = np.zeros(NDAY, dtype=float)
        n_years_eff = 0
        for yi in years_idx:
            d0 = int(yi * NDAY)
            d1 = int(d0 + NDAY)
            if d1 > mhw_class.shape[1]:
                continue
            mhw_y = mhw_class[loc, d0:d1]
            tmp_y = temp_class[loc, d0:d1]
            acc += ((mhw_y == 1) & (tmp_y == group_idx)).astype(float)
            n_years_eff += 1
        out[loc + 1] = (acc / float(n_years_eff)) * 100.0 if n_years_eff > 0 else 0.0

    out[0] = np.mean(out[1:], axis=0)
    return out


def _month_axis_config():
    month_days = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dtype=int)
    month_starts0 = np.concatenate([[0], np.cumsum(month_days)[:-1]])
    month_mid0 = month_starts0 + month_days / 2.0
    month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    return month_days, (month_starts0 + 1).tolist(), (month_mid0 + 1).tolist(), month_labels


def _summarize_block_stats(arr, name):
    flat = np.asarray(arr).reshape(-1)
    finite = flat[np.isfinite(flat)]
    if finite.size == 0:
        print(f"[{name}] no finite values")
        return
    qs = np.percentile(finite, [0, 50, 95, 99, 100])
    print(f"[{name}] min={qs[0]:.3g} p50={qs[1]:.3g} p95={qs[2]:.3g} p99={qs[3]:.3g} max={qs[4]:.3g}")


def _season_doy_mask(month_days, center_months_1based):
    """返回长度365的 bool mask，标记 center_months 覆盖的所有 day-of-year。"""
    month_days = np.asarray(month_days, dtype=int)
    starts0 = np.concatenate([[0], np.cumsum(month_days)[:-1]])
    ends0 = starts0 + month_days  # exclusive
    months = sorted(set(int(m) for m in center_months_1based))
    mask = np.zeros(int(np.sum(month_days)), dtype=bool)
    for m in months:
        if 1 <= m <= 12:
            mask[starts0[m - 1]:ends0[m - 1]] = True
    return mask


def _best_cut_to_center_mask(mask_bool):
    """选择 cut，使 mask 覆盖的窗口在新轴上"窗口中点"对齐到 DOY 中点。

    约束：
    - 先用最大 gap 选择切口，确保跨年季节在新轴上成为连续段；
    - 再平移，使该连续段的中点落在新轴中点。

    返回 cut (0..364)
    """
    n = int(mask_bool.size)
    if n == 0 or not np.any(mask_bool):
        return 0

    idx = np.where(mask_bool)[0]

    gaps = np.diff(np.concatenate([idx, [idx[0] + n]]))
    split_at = int(np.argmax(gaps))
    cut0 = int((idx[split_at] + 1) % n)  # 从最大空档之后开始

    season_new = (idx - cut0) % n
    season_new_sorted = np.sort(season_new)

    start_new = int(season_new_sorted[0])
    end_new = int(season_new_sorted[-1])

    season_mid_new = (start_new + end_new) / 2.0
    target_mid = (n - 1) / 2.0

    shift = target_mid - season_mid_new
    cut = int((cut0 - int(round(shift))) % n)
    return cut


def _center_season_reorder_indices(month_days, center_months_1based):
    """根据给定中心月份(1-12)计算重排索引，使这些月份落在 x 轴中间附近。
    返回 (perm, highlight_ranges)
      perm: 长度365的索引数组，用于 arr[:, perm]
      highlight_ranges: 重排后需要高亮的 x 区间列表 [(x0,x1), ...]，x 用 1..365 轴坐标
    """
    month_days = np.asarray(month_days, dtype=int)
    n = int(np.sum(month_days))
    center_months_1based = [int(m) for m in center_months_1based]
    if len(center_months_1based) == 0:
        perm = np.arange(n, dtype=int)
        return perm, []

    mask = _season_doy_mask(month_days, center_months_1based)
    cut = _best_cut_to_center_mask(mask)
    perm = np.concatenate([np.arange(cut, n, dtype=int), np.arange(0, cut, dtype=int)])

    season_new = (np.where(mask)[0] - cut) % n
    season_new_sorted = np.sort(season_new)
    if season_new_sorted.size == 0:
        return perm, []

    gaps = np.diff(np.concatenate([season_new_sorted, [season_new_sorted[0] + n]]))
    split_at = int(np.argmax(gaps))
    if gaps[split_at] > 1:
        a = int(season_new_sorted[split_at])
        b = int(season_new_sorted[(split_at + 1) % season_new_sorted.size])
        ranges0 = [(0, a), (b, n - 1)]
    else:
        ranges0 = [(int(season_new_sorted[0]), int(season_new_sorted[-1]))]
    highlight_ranges = [(r0 + 1, r1 + 2) for (r0, r1) in ranges0]
    return perm, highlight_ranges


def plot_moving_daily_occurrence_heatmaps_by_location_two_groups_centered():
    periods = [
        (1982, 2001, '1982-2001'),
        (2002, 2022, '2002-2022'),
        (2023, 2043, '2023-2043'),
        (2044, 2064, '2044-2064'),
        (2065, 2085, '2065-2085'),
    ]

    opt_idx = next(i for i, lbl in enumerate(LABELS) if 'optimum' in lbl.lower())
    sev_idx = next(i for i, lbl in enumerate(LABELS) if 'severe' in lbl.lower())
    group_specs = [('Optimum', opt_idx), ('Severe', sev_idx)]

    center_months_by_loc = {
        'Franklin Bay': [11, 12, 1],
        'Western Greenland': [12],
        'Novaya Zemlya': [11],
        'Svalbard': [1, 2, 3],
        'Chukchi Sea': [12, 1, 2],
        'East Greenland': [12, 1, 2, 3],
        'Hudson Bay': [12, 1, 2],
        'Severnaya Zemlya': [9, 10],
    }

    def _norm_loc_key(s):
        return ''.join(str(s).lower().split())

    center_months_by_loc_norm = {_norm_loc_key(k): v for k, v in center_months_by_loc.items()}

    highlight_color = '#0f766e'
    highlight_lw = 3.5

    nc_path = COMPRESSED_FILES['moving']
    month_days, month_starts, month_midpoints, month_labels = _month_axis_config()
    ylabels = [p[2] for p in periods]
    yticks_pos = np.arange(len(periods)) + 0.5

    all_vals = []
    per_loc_blocks = {loc_name: {} for loc_name in LOCATION_NAMES}
    for loc_i, loc_name in enumerate(LOCATION_NAMES):
        for gname, gidx in group_specs:
            rows = []
            for (y0, y1, _) in periods:
                years_idx = np.arange(y0, y1 + 1) - YEAR0
                block9 = _daily_occurrence_frequency_percent(nc_path, years_idx, gidx)
                rows.append(np.clip(block9[loc_i + 1], 0.0, 100.0))
            mat = np.stack(rows, axis=0)
            per_loc_blocks[loc_name][gname] = mat
            all_vals.append(mat.reshape(-1))

    all_vals = np.concatenate(all_vals) if all_vals else np.array([])
    vmax = float(np.nanmax(all_vals)) if all_vals.size else 1.0
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    vmax = min(vmax, 20.0)
    vmin = 0.0

    _summarize_block_stats(all_vals, "new-occurrence (optimum+severe, centered)")

    for loc_name in LOCATION_NAMES:
        center_months = center_months_by_loc.get(loc_name)
        if center_months is None:
            center_months = center_months_by_loc_norm.get(_norm_loc_key(loc_name), [])
        perm, highlight_ranges = _center_season_reorder_indices(month_days, center_months)

        cut = int(perm[0])

        def _map_x(old_x1):
            return int(((int(old_x1) - 1 - cut) % NDAY) + 1)

        month_mid_new = [_map_x(x) for x in month_midpoints]
        month_starts_new = [_map_x(x) for x in month_starts]
        order = np.argsort(month_mid_new)
        month_mid_new_sorted = [month_mid_new[i] for i in order]
        month_labels_sorted = [month_labels[i] for i in order]

        fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.2), sharey=True, constrained_layout=True)
        last_im = None
        for ax, (gname, _) in zip(axes, group_specs):
            data = per_loc_blocks[loc_name][gname][:, perm]
            im = ax.imshow(
                data,
                aspect='auto',
                cmap='YlOrRd',
                vmin=vmin,
                vmax=vmax,
                extent=[1, NDAY + 1, len(periods), 0],
                interpolation='nearest',
            )
            last_im = im
            ax.set_title(gname, fontsize=11, fontweight='bold')
            ax.set_xlim(1, NDAY + 1)
            ax.set_ylim(len(periods), 0)
            ax.set_xticks(month_mid_new_sorted)
            ax.set_xticklabels(month_labels_sorted, fontsize=9)
            ax.set_xlabel('Month', fontsize=10)
            ax.tick_params(axis='both', labelsize=9)

            for ms in month_starts_new[1:]:
                ax.axvline(x=ms, color='white', linewidth=0.8, alpha=0.25)

            # 高亮月份窗口：仅画不透明边框，不使用任何半透明背景
            for (x0, x1) in highlight_ranges:
                ax.vlines(
                    [x0, x1],
                    ymin=0,
                    ymax=len(periods),
                    colors=highlight_color,
                    linewidth=highlight_lw,
                    zorder=6,
                )

        axes[0].set_yticks(yticks_pos)
        axes[0].set_yticklabels(ylabels, fontsize=9)
        axes[0].set_ylabel('Period', fontsize=10)

        fig.suptitle(f"{loc_name}", fontsize=13, fontweight='bold')
        cbar = fig.colorbar(last_im, ax=axes, location='right', shrink=0.92, pad=0.02)
        cbar.set_label('% of years', fontsize=10)
        cbar.ax.tick_params(labelsize=9)
        plt.show()


plot_moving_daily_occurrence_heatmaps_by_location_two_groups_centered()

In [ ]:
# === Test: Arctic map as small locator + big heatmaps around it ===
# 进一步优化：
# - 严格避免缩略图互相遮挡：离散候选槽位 + 贪心无碰撞放置（必要时缩小 zoom / 增加外圈）
# - 引线连接到热图边框：从地图端点 -> 热图边框最近点，可折线
# - 缩略图 yticks 强制显示（仅左子图），且不显示 y 轴标题

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox


def _render_loc_heatmap_thumbnail(
    loc_name,
    per_loc_blocks,
    group_specs,
    month_days,
    month_midpoints,
    month_starts,
    month_labels,
    periods,
    ylabels,
    vmax=20.0,
    vmin=0.0,
    cmap='YlOrRd',
    font_delta=8,
    xtick_stride=2,
):
    """把某海区的 Optimum/Severe 并排热图渲染成一张 RGBA 图片数组（用于贴图）。"""

    center_months_by_loc = {
        'Franklin Bay': [11, 12, 1],
        'Western Greenland': [12],
        'Novaya Zemlya': [11],
        'Svalbard': [1, 2, 3],
        'Chukchi Sea': [12, 1, 2],
        'East Greenland': [12, 1, 2, 3],
        'Hudson Bay': [12, 1, 2],
        'Severnaya Zemlya': [9, 10],
    }

    def _norm_loc_key(s):
        return ''.join(str(s).lower().split())

    key_norm = _norm_loc_key(loc_name)
    center_months_by_loc_norm = {_norm_loc_key(k): v for k, v in center_months_by_loc.items()}
    center_months = center_months_by_loc.get(loc_name)
    if center_months is None:
        center_months = center_months_by_loc_norm.get(key_norm, [])

    perm, highlight_ranges = _center_season_reorder_indices(month_days, center_months)
    cut = int(perm[0])

    def _map_x(old_x1):
        return int(((int(old_x1) - 1 - cut) % NDAY) + 1)

    month_mid_new = [_map_x(x) for x in month_midpoints]
    month_starts_new = [_map_x(x) for x in month_starts]
    order = np.argsort(month_mid_new)
    month_mid_new_sorted = [month_mid_new[i] for i in order]
    month_labels_sorted = [month_labels[i] for i in order]

    if xtick_stride is None or int(xtick_stride) <= 1:
        xticks = month_mid_new_sorted
        xlabels = month_labels_sorted
    else:
        stride = int(xtick_stride)
        xticks = month_mid_new_sorted[::stride]
        xlabels = month_labels_sorted[::stride]

    highlight_color = '#0f766e'
    highlight_lw = 3.5

    fs_title = 8 + font_delta
    fs_xtick = 6 + font_delta
    fs_suptitle = 10 + font_delta
    fs_ytick = 9 + font_delta

    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.9), sharey=True)
    fig.subplots_adjust(left=0.20, right=0.99, bottom=0.18, top=0.86, wspace=0.08)

    yticks_pos = np.arange(len(periods)) + 0.5

    for k, (ax, (gname, _)) in enumerate(zip(axes, group_specs)):
        data = per_loc_blocks[loc_name][gname][:, perm]
        ax.imshow(
            data,
            aspect='auto',
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            extent=[1, NDAY + 1, len(periods), 0],
            interpolation='nearest',
        )
        ax.set_title(gname, fontsize=fs_title, fontweight='bold', pad=4)
        ax.set_xlim(1, NDAY + 1)
        ax.set_ylim(len(periods), 0)
        ax.set_xticks(xticks)
        ax.set_xticklabels(xlabels, fontsize=fs_xtick)
        ax.tick_params(axis='x', length=0)

        ax.set_yticks(yticks_pos)
        if k == 0:
            ax.set_yticklabels(ylabels, fontsize=fs_ytick)
            ax.set_ylabel('')
        else:
            ax.set_yticklabels([])

        ax.yaxis.set_visible(True)
        ax.tick_params(axis='y', which='both', left=True, labelleft=(k == 0))

        for ms in month_starts_new[1:]:
            ax.axvline(x=ms, color='white', linewidth=0.8, alpha=0.25)

        for (x0, x1) in highlight_ranges:
            ax.vlines([x0, x1], ymin=0, ymax=len(periods), colors=highlight_color, linewidth=highlight_lw, zorder=6)

    fig.suptitle(loc_name, fontsize=fs_suptitle, fontweight='bold', y=0.98)

    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
    plt.close(fig)
    return img


def _rects_overlap(c1, c2, hw):
    hx, hy = hw
    return (abs(c1[0] - c2[0]) < 2 * hx) and (abs(c1[1] - c2[1]) < 2 * hy)


def _candidate_slots(bounds, ring_radii, n_theta, phase_offset=0.0, center=(0.5, 0.5)):
    xmin, xmax, ymin, ymax = bounds
    cx, cy = center
    slots = []
    for r in ring_radii:
        for k in range(n_theta):
            theta = phase_offset + 2 * np.pi * (k / n_theta)
            x = cx + r * np.cos(theta)
            y = cy + r * np.sin(theta)
            x = float(np.clip(x, xmin, xmax))
            y = float(np.clip(y, ymin, ymax))
            slots.append((x, y))
    return slots


def _strict_place_no_overlap(anchors, slots, box_hw, prefer_center=(0.5, 0.5)):
    anchors = np.array(anchors, dtype=float)
    cx, cy = prefer_center

    d0 = np.hypot(anchors[:, 0] - cx, anchors[:, 1] - cy)
    order = np.argsort(d0)

    assigned = [None] * len(anchors)
    placed = []

    for ii in order:
        xa, ya = anchors[ii]
        best = None
        best_cost = None

        for s in slots:
            ok = True
            for p in placed:
                if _rects_overlap(s, p, box_hw):
                    ok = False
                    break
            if not ok:
                continue

            da = np.hypot(s[0] - xa, s[1] - ya)
            dc = np.hypot(s[0] - cx, s[1] - cy)
            cost = da + 0.18 * (1.0 / (dc + 1e-6))

            if best_cost is None or cost < best_cost:
                best_cost = cost
                best = s

        if best is None:
            return None

        assigned[ii] = best
        placed.append(best)

    return assigned


def _nearest_point_on_rect(center, target, hw):
    cx, cy = center
    tx, ty = target
    hx, hy = hw

    x = float(np.clip(tx, cx - hx, cx + hx))
    y = float(np.clip(ty, cy - hy, cy + hy))

    inside = (cx - hx <= x <= cx + hx) and (cy - hy <= y <= cy + hy)
    if inside:
        dl = abs(x - (cx - hx))
        dr = abs((cx + hx) - x)
        db = abs(y - (cy - hy))
        dt = abs((cy + hy) - y)
        m = min(dl, dr, db, dt)
        if m == dl:
            x = cx - hx
        elif m == dr:
            x = cx + hx
        elif m == db:
            y = cy - hy
        else:
            y = cy + hy

    return (x, y)


def test_arctic_map_with_big_heatmaps_and_global_colorbar():
    periods = [
        (1982, 2001, '1982-2001'),
        (2002, 2022, '2002-2022'),
        (2023, 2043, '2023-2043'),
        (2044, 2064, '2044-2064'),
        (2065, 2085, '2065-2085'),
    ]
    ylabels = [p[2] for p in periods]

    opt_idx = next(i for i, lbl in enumerate(LABELS) if 'optimum' in lbl.lower())
    sev_idx = next(i for i, lbl in enumerate(LABELS) if 'severe' in lbl.lower())
    group_specs = [('Optimum', opt_idx), ('Severe', sev_idx)]

    nc_path = COMPRESSED_FILES['moving']

    month_days, month_starts, month_midpoints, month_labels = _month_axis_config()

    per_loc_blocks = {loc_name: {} for loc_name in LOCATION_NAMES}
    all_vals = []
    for loc_i, loc_name in enumerate(LOCATION_NAMES):
        for gname, gidx in group_specs:
            rows = []
            for (y0, y1, _) in periods:
                years_idx = np.arange(y0, y1 + 1) - YEAR0
                block9 = _daily_occurrence_frequency_percent(nc_path, years_idx, gidx)
                rows.append(np.clip(block9[loc_i + 1], 0.0, 100.0))
            mat = np.stack(rows, axis=0)
            per_loc_blocks[loc_name][gname] = mat
            all_vals.append(mat.reshape(-1))

    all_vals = np.concatenate(all_vals) if all_vals else np.array([])
    vmax = float(np.nanmax(all_vals)) if all_vals.size else 1.0
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    vmax = min(vmax, 20.0)
    vmin = 0.0
    cmap = 'YlOrRd'

    fig = plt.figure(figsize=(16.0, 11.0))
    fig.suptitle('Arctic locator map + big per-location heatmaps', fontsize=18, fontweight='bold', y=0.98)

    map_rect = [0.42, 0.42, 0.16, 0.16]

    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.io import shapereader
    import shapely.geometry as sgeom
    from shapely.ops import unary_union

    proj = ccrs.NorthPolarStereo()
    axm = fig.add_axes(map_rect, projection=proj)
    axm.set_extent([-180, 180, 50, 90], crs=ccrs.PlateCarree())

    gl = axm.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.6,
        color='gray',
        alpha=0.55,
        linestyle='--',
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = False
    gl.xlocator = plt.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = plt.FixedLocator(np.arange(50, 91, 10))

    lons = np.linspace(-180, 180, 721)
    axm.plot(lons, np.full_like(lons, 50.0), transform=ccrs.PlateCarree(), color='black', linewidth=2.2, zorder=40)

    land_shp = shapereader.natural_earth(resolution='110m', category='physical', name='land')
    land_geoms = list(shapereader.Reader(land_shp).geometries())
    land_union = unary_union(land_geoms)

    boxes = [
        ('Franklin Bay',      -128, -125, 69, 72),
        ('Western Greenland',  -54,  -51, 70, 73),
        ('Novaya Zemlya',       30,   70, 63, 76),
        ('Svalbard',            15,   30, 76, 81),
        ('ChukchiSea',        -178, -160, 67, 72),
        ('East Greenland',     -40,  -15, 67, 76),
        ('Hudson Bay',         -95,  -74, 51, 64),
        ('Severnaya Zemlya',    90,  108, 77, 82),
    ]

    fill_color = '#2dd4bf'
    edge_color = '#0f766e'

    ocean_geoms_by_name = {}
    for name, lon_w, lon_e, lat_s, lat_n in boxes:
        ocean_parts = []
        for lons_poly, lats_poly in _box_polygons(lon_w, lon_e, lat_s, lat_n):
            box_poly = sgeom.Polygon(zip(lons_poly, lats_poly))
            ocean_poly = box_poly.difference(land_union)
            if ocean_poly.is_empty:
                continue
            ocean_parts.append(ocean_poly)
            axm.add_geometries([ocean_poly], crs=ccrs.PlateCarree(), facecolor=fill_color, edgecolor='none', alpha=0.28, zorder=3)
        if ocean_parts:
            ocean_geoms_by_name[name] = unary_union(ocean_parts)

    axm.add_feature(cfeature.LAND.with_scale('110m'), facecolor='#d9d9d9', edgecolor='none', zorder=6)
    axm.add_feature(cfeature.COASTLINE.with_scale('110m'), linewidth=0.9, zorder=7)

    _raise_gridlines_zorder(gl, zorder=55)

    anchors_fig = []
    names_for_thumbs = []
    map_tip_fig = []

    for name, lon_w, lon_e, lat_s, lat_n in boxes:
        ocean_geom = ocean_geoms_by_name.get(name)
        if ocean_geom is None or ocean_geom.is_empty:
            continue

        for seg_lons, seg_lats in _iter_lonlat_lines_from_boundary(ocean_geom):
            axm.plot(seg_lons, seg_lats, transform=ccrs.PlateCarree(), color=edge_color, linewidth=2.0, linestyle='-', zorder=50)

        axm._tmp_ocean_boundary = _ocean_boundary_to_axes(axm, ocean_geom)

        lon_c, lat_c = float(ocean_geom.centroid.x), float(ocean_geom.centroid.y)
        xA0, yA0 = axm.transAxes.inverted().transform(
            axm.transData.transform(axm.projection.transform_point(lon_c, lat_c, ccrs.PlateCarree()))
        )
        dx, dy = xA0 - 0.5, yA0 - 0.5
        dn = (dx * dx + dy * dy) ** 0.5
        if not np.isfinite(dn) or dn == 0:
            continue
        dx, dy = dx / dn, dy / dn

        xB, yB = 0.5 + 0.5 * dx, 0.5 + 0.5 * dy
        r_label = 0.92
        xT, yT = 0.5 + r_label * dx, 0.5 + r_label * dy

        best = _radial_outer_boundary_point_axes(axm, lon_c, lat_c)
        if best is not None:
            xA, yA = best
        else:
            xA, yA = xA0, yA0

        axm.plot([xA, xB, xT], [yA, yB, yT], transform=axm.transAxes, color=edge_color, linewidth=1.6, linestyle=(0, (4, 3)), zorder=60, clip_on=False)

        x_disp, y_disp = axm.transAxes.transform((xT, yT))
        x_fig, y_fig = fig.transFigure.inverted().transform((x_disp, y_disp))
        anchors_fig.append((float(x_fig), float(y_fig)))
        map_tip_fig.append((float(x_fig), float(y_fig)))
        names_for_thumbs.append(name)

    if hasattr(axm, '_tmp_ocean_boundary'):
        delattr(axm, '_tmp_ocean_boundary')

    _set_circular_boundary(axm)
    axm.set_title('Arctic locator', fontsize=10, fontweight='bold', pad=4)

    def _norm(s):
        return ''.join(str(s).lower().split())

    # 尝试多种 zoom：从大到小，直到找到严格无重叠方案
    zoom_candidates = [0.52, 0.50, 0.48, 0.46]

    # 先渲染一次图片，用于估计 box_hw
    sample_loc = LOCATION_NAMES[0]
    sample_img = _render_loc_heatmap_thumbnail(
        sample_loc,
        per_loc_blocks,
        group_specs,
        month_days,
        month_midpoints,
        month_starts,
        month_labels,
        periods,
        ylabels,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        font_delta=8,
        xtick_stride=2,
    )

    fig_w_px = fig.get_figwidth() * fig.dpi
    fig_h_px = fig.get_figheight() * fig.dpi

    placed = None
    chosen_zoom = None
    chosen_box_hw = None

    for zoom in zoom_candidates:
        img_h, img_w = sample_img.shape[0], sample_img.shape[1]
        # figure 坐标中的半宽高（加上保守 padding，确保“视觉不相交”）
        hw = 0.5 * (img_w * zoom / fig_w_px)
        hh = 0.5 * (img_h * zoom / fig_h_px)
        pad = 0.012
        box_hw = (hw + pad, hh + pad)

        bounds = (box_hw[0] + 0.02, 0.90 - box_hw[0], box_hw[1] + 0.06, 0.92 - box_hw[1])
        ring_radii = [0.34, 0.44, 0.54, 0.64, 0.72, 0.80, 0.88]
        slots = _candidate_slots(bounds, ring_radii=ring_radii, n_theta=84, phase_offset=0.20)
        placed = _strict_place_no_overlap(anchors_fig, slots, box_hw=box_hw)
        if placed is not None:
            chosen_zoom = zoom
            chosen_box_hw = box_hw
            break

    if placed is None:
        raise RuntimeError('Cannot find non-overlapping placement even after reducing zoom.')

    zoom = chosen_zoom
    box_hw = chosen_box_hw
    xy_adj_fig = placed

    thumb_centers = {}

    for name, center_fig in zip(names_for_thumbs, xy_adj_fig):
        loc_key = name
        if loc_key not in per_loc_blocks:
            match = next((ln for ln in per_loc_blocks.keys() if _norm(ln) == _norm(loc_key)), None)
            if match is None:
                continue
            loc_key = match

        img = _render_loc_heatmap_thumbnail(
            loc_key,
            per_loc_blocks,
            group_specs,
            month_days,
            month_midpoints,
            month_starts,
            month_labels,
            periods,
            ylabels,
            vmin=vmin,
            vmax=vmax,
            cmap=cmap,
            font_delta=8,
            xtick_stride=2,
        )

        imagebox = OffsetImage(img, zoom=zoom)
        ab = AnnotationBbox(
            imagebox,
            center_fig,
            xycoords=fig.transFigure,
            frameon=True,
            bboxprops=dict(boxstyle='round,pad=0.25', fc='white', ec='#0f766e', lw=2.0),
            zorder=80,
            clip_on=False,
        )
        fig.add_artist(ab)
        thumb_centers[name] = tuple(center_fig)

    # === 连线：map_tip -> 缩略图边框 ===
    for name, p0 in zip(names_for_thumbs, map_tip_fig):
        center = thumb_centers.get(name)
        if center is None:
            continue
        p1 = _nearest_point_on_rect(center, p0, box_hw)

        x0, y0 = p0
        mx = 0.70 * x0 + 0.30 * p1[0]
        my = 0.70 * y0 + 0.30 * p1[1]

        fig.lines.append(
            plt.Line2D(
                [x0, mx, p1[0]],
                [y0, my, p1[1]],
                transform=fig.transFigure,
                color='#0f766e',
                linewidth=1.6,
                linestyle=(0, (4, 3)),
                zorder=95,
            )
        )

    sm = plt.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])

    cax = fig.add_axes([0.92, 0.18, 0.020, 0.64])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label('% of years', fontsize=16, fontweight='bold')
    cbar.ax.tick_params(labelsize=14)
    filename='bigmap2.svg'
    print(f"Saving {filename} ...")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Saved {filename} successfully.")
    plt.show()


test_arctic_map_with_big_heatmaps_and_global_colorbar()